In [1]:
import pandas as pd
import os

In [2]:
os.makedirs('data_clean', exist_ok=True)

# Download data 

In [3]:
orders = pd.read_csv('data/olist_orders_dataset.csv', encoding='utf-8')
order_items = pd.read_csv('data/olist_order_items_dataset.csv', encoding='utf-8')
payments = pd.read_csv('data/olist_order_payments_dataset.csv', encoding='utf-8')
products = pd.read_csv('data/olist_products_dataset.csv', encoding='utf-8')
translation = pd.read_csv('data/product_category_name_translation.csv', encoding='utf-8')
reviews = pd.read_csv('data/olist_order_reviews_dataset.csv', encoding='utf-8')
customers = pd.read_csv('data/olist_customers_dataset.csv', encoding='utf-8', dtype={'customer_zip_code_prefix': str})
sellers = pd.read_csv('data/olist_sellers_dataset.csv', encoding='utf-8', dtype={'seller_zip_code_prefix': str})
geolocation = pd.read_csv('data/olist_geolocation_dataset.csv', encoding='utf-8', dtype={'geolocation_zip_code_prefix': str})

# DATA TRANSFORMATION & CLEANING 

# Adjust Date and create Calendar Table

In [4]:

date_columns = [
    'order_purchase_timestamp', 'order_approved_at', 
    'order_delivered_carrier_date', 'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]
for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])


min_date = orders['order_purchase_timestamp'].min().normalize()
max_date = orders['order_purchase_timestamp'].max().normalize()
date_range = pd.date_range(start=min_date, end=max_date, freq='D')
calendar = pd.DataFrame({'date': date_range})
calendar['year'] = calendar['date'].dt.year
calendar['month'] = calendar['date'].dt.month
calendar['quarter'] = calendar['date'].dt.quarter
calendar['weekday'] = calendar['date'].dt.dayofweek

# Manage Products

In [5]:

products_cleaned = pd.merge(products, translation, on='product_category_name', how='left')
products_cleaned['product_category_name_english'] = products_cleaned['product_category_name_english'].fillna('unknown')
for col in ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']:
    products_cleaned[col] = pd.to_numeric(products_cleaned[col], errors='coerce')

# Change Order Items info

In [6]:

order_items['price'] = pd.to_numeric(order_items['price'], errors='coerce')
order_items['freight_value'] = pd.to_numeric(order_items['freight_value'], errors='coerce')

if (order_items['price'] < 0).any() or (order_items['freight_value'] < 0).any():
    order_items.loc[order_items['price'] < 0, 'price'] = 0
    order_items.loc[order_items['freight_value'] < 0, 'freight_value'] = 0

orphan_items = order_items[~order_items['product_id'].isin(products_cleaned['product_id'])]
if len(orphan_items) > 0:
    print(f"⚠️ Found {len(orphan_items)} orphan order items. Proceeding with caution.")

# Fix Geolocation 

In [7]:

geo_cleaned = geolocation.groupby('geolocation_zip_code_prefix').agg({
    'geolocation_lat': 'mean',
    'geolocation_lng': 'mean',
    'geolocation_city': 'first',
    'geolocation_state': 'first'
}).reset_index()


# Calcurate Delivery Days & Impute

In [8]:

orders['delivery_days_actual'] = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.days
delivered_mask = (orders['order_status'] == 'delivered') & (orders['delivery_days_actual'].isna())
if delivered_mask.any():
    orders.loc[delivered_mask, 'delivery_days_actual'] = (
        orders.loc[delivered_mask, 'order_delivered_carrier_date'] - 
        orders.loc[delivered_mask, 'order_purchase_timestamp']
    ).dt.days



# Manage Payments 

In [9]:

high_val_thresh = payments['payment_value'].quantile(0.99)
payments['is_high_value'] = payments['payment_value'] > high_val_thresh

payment_per_order = payments.groupby('order_id', as_index=False).agg({
    'payment_value': 'sum',
    'payment_installments': 'sum'
})
payment_per_order.rename(columns={'payment_value': 'total_payment_value'}, inplace=True)
orders = orders.merge(payment_per_order, on='order_id', how='left')


# Drop Reviews 

In [10]:

reviews = reviews.drop_duplicates(subset=['order_id'], keep='first')


In [11]:

orders.to_csv('data_clean/orders_clean.csv', index=False, encoding='utf-8')
order_items.to_csv('data_clean/order_items_clean.csv', index=False, encoding='utf-8')
payments.to_csv('data_clean/payments_clean.csv', index=False, encoding='utf-8')
products_cleaned.to_csv('data_clean/products_clean.csv', index=False, encoding='utf-8')
reviews.to_csv('data_clean/reviews_clean.csv', index=False, encoding='utf-8')
customers.to_csv('data_clean/customers_clean.csv', index=False, encoding='utf-8')
sellers.to_csv('data_clean/sellers_clean.csv', index=False, encoding='utf-8')
geo_cleaned.to_csv('data_clean/geolocation_clean.csv', index=False, encoding='utf-8')
calendar.to_csv('data_clean/calendar_clean.csv', index=False, encoding='utf-8') # Export Calendar Table

